# Pseudo-Trial Averaging: Disjoint vs Bootstrap Comparison

Compares SNR-boosting strategies for stimulus identity decoding (A_stim, 4-class) from rule1 MEG epochs.

**Configurations tested:**
- Single-trial baseline (`k=1`)
- Disjoint averaging: `k ∈ {3, 5}`, R=20 repetitions
- Bootstrap averaging: `k ∈ {3, 5}`, M matched to disjoint count, R=20 repetitions
- Bootstrap `k=5`, M inflated to `4 × N_c/k`

All configurations use `pseudo_test=False` (train on pseudo-trials, evaluate on single trials), which is the asymmetric option recommended when test folds are small.

In [ ]:
import sys
import warnings
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

sys.path.insert(0, str(Path('..').resolve()))
from toolkit import (
    RawAmplitudeFeature,
    RidgeLogisticDecoder,
    CVSplitter,
    cross_validate,
    make_pseudo_trials,
)

print('toolkit loaded')

## 1. Load data

In [ ]:
EPOCHS_PATH = Path('../ported_results/sub-001_ses-01_task-binding_epo.fif')
BEHAV_PATH  = Path('/mnt/storage/NEU502B/brain-binding/data/2026-04-10/sub-001_events.csv')

epochs = mne.read_epochs(str(EPOCHS_PATH), preload=True)
df     = pd.read_csv(BEHAV_PATH).sort_values('trial').reset_index(drop=True)

print(f'Epochs : {epochs}')
print(f'Trials : {len(df)}')
print(f'A_stim categories : {sorted(df["A_stim"].unique())}')

In [ ]:
# Extract rule1 epochs (offset 1 within each 9-event trial block)
# STAGE_OFFSETS: fixation=0, rule1=1, rule2=2, rule3=3, transition=4,
#                test1=5, test2=6, test3=7, response=8
n_trials = len(df)
rule1_idx = [9 * t + 1 for t in range(n_trials)]
ep_rule1  = epochs[rule1_idx]

X    = ep_rule1.get_data(picks='meg').astype(np.float32)  # (n_trials, n_ch, n_t)
sfreq = ep_rule1.info['sfreq']
times = ep_rule1.times

STIM_CATEGORIES = ['circle', 'rectangle', 'star', 'triangle']
cat_map = {c: i for i, c in enumerate(STIM_CATEGORIES)}
y = df['A_stim'].map(cat_map).values.astype(int)

print(f'X shape   : {X.shape}   (trials × channels × timepoints)')
print(f'sfreq     : {sfreq} Hz')
print(f'Times     : {times[0]:.3f} – {times[-1]:.3f} s')
print(f'Labels    : {dict(zip(*np.unique(y, return_counts=True)))}')

## 2. Quick sanity check: pseudo-trial construction

In [ ]:
# Use first 80% of data as a proxy training fold to inspect pseudo-trial shapes
n_tr = int(0.8 * len(X))
X_tr, y_tr = X[:n_tr], y[:n_tr]

n_per_class = np.bincount(y_tr)
print('Trials per class in train proxy:', n_per_class)

for k in (3, 5):
    for mode in ('disjoint', 'bootstrap'):
        Xp, yp = make_pseudo_trials(X_tr, y_tr, k=k, mode=mode,
                                    rng=np.random.default_rng(0))
        print(f'  k={k} {mode:9s} → Xp {Xp.shape}  '
              f'per-class: {np.bincount(yp)}')

## 3. Define experiment configurations

In [ ]:
# Approximate training fold size for N_c/k baseline estimate.
# With 5-fold CV: ~80% of ~30 trials/class ≈ 24 trials/class in training.
# disjoint k=5: 24//5 = 4 pseudo/class
# inflated bootstrap: 4 * 4 = 16 pseudo/class

CONFIGS = [
    dict(
        label     = 'single-trial (k=1)',
        pseudo_k  = 1,
        color     = 'black',
        ls        = '-',
        lw        = 2.0,
    ),
    dict(
        label              = 'disjoint k=3, R=20',
        pseudo_k           = 3,
        pseudo_mode        = 'disjoint',
        pseudo_repetitions = 20,
        color              = '#1f77b4',
        ls                 = '-',
        lw                 = 1.5,
    ),
    dict(
        label              = 'disjoint k=5, R=20',
        pseudo_k           = 5,
        pseudo_mode        = 'disjoint',
        pseudo_repetitions = 20,
        color              = '#2ca02c',
        ls                 = '-',
        lw                 = 1.5,
    ),
    dict(
        label              = 'bootstrap k=3, M=N_c/k, R=20',
        pseudo_k           = 3,
        pseudo_mode        = 'bootstrap',
        pseudo_n_per_class = None,   # matches disjoint count
        pseudo_repetitions = 20,
        color              = '#1f77b4',
        ls                 = '--',
        lw                 = 1.5,
    ),
    dict(
        label              = 'bootstrap k=5, M=N_c/k, R=20',
        pseudo_k           = 5,
        pseudo_mode        = 'bootstrap',
        pseudo_n_per_class = None,   # matches disjoint count
        pseudo_repetitions = 20,
        color              = '#2ca02c',
        ls                 = '--',
        lw                 = 1.5,
    ),
    dict(
        label              = 'bootstrap k=5, M=4×N_c/k, R=20',
        pseudo_k           = 5,
        pseudo_mode        = 'bootstrap',
        # Inflated: 4× the disjoint count per class ≈ 4 * (24//5) = 16
        # We use None here and set n_pseudo_per_class per fold dynamically.
        # Since cross_validate applies the same M to all folds, we pre-compute
        # a reasonable fixed value: floor(0.8 * min_class_count / 5) * 4
        pseudo_n_per_class = 16,
        pseudo_repetitions = 20,
        color              = '#d62728',
        ls                 = ':',
        lw                 = 2.0,
    ),
]

print(f'{len(CONFIGS)} configurations to run')

## 4. Run all configurations

In [ ]:
feature      = RawAmplitudeFeature(time_window=(-0.05, 0.4), standardize=True)
decoder_fac  = lambda: RidgeLogisticDecoder(C=0.1, max_iter=500)
cv           = CVSplitter(n_splits=5, stratified=True, random_state=42)

results = {}

for cfg in CONFIGS:
    label = cfg['label']
    print(f'Running: {label} ...', flush=True)

    kw = dict(
        feature         = feature,
        decoder_factory = decoder_fac,
        X               = X,
        y               = y,
        sfreq           = sfreq,
        times           = times,
        cv              = cv,
        metric          = 'roc_auc',
        pseudo_k        = cfg.get('pseudo_k', 1),
        pseudo_mode     = cfg.get('pseudo_mode', 'disjoint'),
        pseudo_n_per_class = cfg.get('pseudo_n_per_class', None),
        pseudo_repetitions = cfg.get('pseudo_repetitions', 1),
        pseudo_seed     = 42,
        pseudo_test     = False,   # asymmetric: test on single trials
    )
    results[label] = cross_validate(**kw)
    r = results[label]
    print(f'  grand mean AUC = {r.mean_scores.mean():.4f}  '
          f'  scores shape = {r.scores.shape}')

## 5. Time-course comparison plot

In [ ]:
# Reconstruct the feature time axis (after time_window + no decimation)
feat_times = times[(times >= -0.05) & (times <= 0.4)]

fig, ax = plt.subplots(figsize=(12, 5))

for cfg in CONFIGS:
    label  = cfg['label']
    r      = results[label]
    mean   = r.mean_scores          # (n_timepoints,)
    sem    = r.sem_scores            # (n_timepoints,)

    ax.plot(feat_times, mean,
            label=label,
            color=cfg['color'],
            linestyle=cfg['ls'],
            linewidth=cfg['lw'])
    ax.fill_between(feat_times, mean - sem, mean + sem,
                    alpha=0.12, color=cfg['color'])

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.9, label='chance (0.5)')
ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ROC-AUC (mean ± SEM across folds)')
ax.set_title('A_stim decoding from rule1 epochs — pseudo-trial averaging comparison')
ax.legend(loc='upper right', fontsize=9)
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.01))
ax.grid(True, which='major', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary bar plot (time-averaged AUC per config)

In [ ]:
# Time-average AUC over the post-stimulus window (0–0.4 s)
post_mask = feat_times >= 0.0

labels_ordered = [cfg['label'] for cfg in CONFIGS]
means_tavg = []
sems_tavg  = []

for cfg in CONFIGS:
    r = results[cfg['label']]
    fold_means = r.scores[:, post_mask].mean(axis=1)   # (n_folds,)
    means_tavg.append(fold_means.mean())
    sems_tavg.append(fold_means.std() / np.sqrt(len(fold_means)))

colors_bar = [cfg['color'] for cfg in CONFIGS]
x = np.arange(len(CONFIGS))

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(x, means_tavg, yerr=sems_tavg,
              color=colors_bar, alpha=0.75,
              capsize=4, width=0.6)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels([c['label'] for c in CONFIGS], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Time-avg ROC-AUC (mean ± SEM across folds)')
ax.set_title('A_stim decoding — post-stimulus average (0–0.4 s)')
ax.set_ylim(0.45, None)
plt.tight_layout()
plt.show()

print('\n--- Grand mean AUC (post-stimulus) ---')
for cfg, m, s in zip(CONFIGS, means_tavg, sems_tavg):
    print(f"  {cfg['label']:<45s}  {m:.4f} ± {s:.4f}")

## 7. Fold-level variance: disjoint vs bootstrap at matched M

In [ ]:
# Compare per-fold AUC distributions for the k=5 head-to-head
k5_labels = [
    'disjoint k=5, R=20',
    'bootstrap k=5, M=N_c/k, R=20',
    'bootstrap k=5, M=4×N_c/k, R=20',
]

fig, axes = plt.subplots(1, len(k5_labels), figsize=(12, 4), sharey=True)

for ax, lbl in zip(axes, k5_labels):
    r = results[lbl]
    fold_tavg = r.scores[:, post_mask].mean(axis=1)   # (n_folds,)
    ax.boxplot(fold_tavg, vert=True, widths=0.4,
               medianprops=dict(color='black', linewidth=2),
               patch_artist=True,
               boxprops=dict(facecolor='lightblue', alpha=0.7))
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(lbl, fontsize=9)
    ax.set_ylabel('Time-avg ROC-AUC')
    ax.set_xticks([])

fig.suptitle('Per-fold AUC distribution: k=5 configurations', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Headline comparison: disjoint vs bootstrap at matched M

Side-by-side time courses for the clean disjoint/bootstrap head-to-head (k=5, matched M).

In [ ]:
headline_cfgs = [
    ('single-trial (k=1)',            'black',   '-',  2.0),
    ('disjoint k=5, R=20',            '#2ca02c', '-',  1.8),
    ('bootstrap k=5, M=N_c/k, R=20',  '#2ca02c', '--', 1.8),
    ('bootstrap k=5, M=4×N_c/k, R=20','#d62728', ':',  2.0),
]

fig, ax = plt.subplots(figsize=(11, 4))

for lbl, col, ls, lw in headline_cfgs:
    r    = results[lbl]
    mean = r.mean_scores
    sem  = r.sem_scores
    ax.plot(feat_times, mean, label=lbl, color=col, linestyle=ls, linewidth=lw)
    ax.fill_between(feat_times, mean - sem, mean + sem, alpha=0.12, color=col)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.9)
ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Time (s)')
ax.set_ylabel('ROC-AUC')
ax.set_title('Disjoint vs Bootstrap (k=5) — A_stim from rule1')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 9. Summary

Key questions answered by this notebook:

1. **Does pseudo-trial averaging help at all?** Compare any pseudo config vs `single-trial (k=1)`.
2. **Disjoint vs bootstrap at matched M?** Compare `disjoint k=5` vs `bootstrap k=5, M=N_c/k`. If they're equivalent, prefer disjoint (simpler, guaranteed independence).
3. **Does inflating M in bootstrap add anything?** Compare `bootstrap k=5, M=N_c/k` vs `bootstrap k=5, M=4×N_c/k`. If not, stick with matched M.
4. **k=3 vs k=5?** More averaging → higher SNR but fewer pseudo-trials per fold. Optimal k is dataset-dependent.

Interpretation notes:
- All configs train on pseudo-trials but evaluate on single trials (`pseudo_test=False`), so they are comparable on the same test distribution.
- SEM here is across CV folds (n=5), not across subjects. Variance is dominated by fold assignment, not repetition randomness.
- The R=20 repetitions reduce variance from unlucky groupings but don't increase the effective sample size.